# 03 — Model Experiments

**Goal:** Train each anomaly detection model, understand how it works, visualize its detections, and identify strengths/weaknesses.

We'll examine each model individually before combining them in an ensemble:

1. **Naive Baseline** — simple threshold rules (the bar to beat)
2. **Statistical (Z-score + EWMA)** — classical control chart methods
3. **Isolation Forest** — tree-based unsupervised anomaly detection
4. **Ensemble** — weighted combination

*(LSTM Autoencoder requires TensorFlow and longer training — covered separately in `scripts/train_models.py`)*

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import DataFetcher
from src.data.feature_engineer import FeatureEngineer
from src.models.naive_baseline import NaiveBaselineDetector
from src.models.statistical import StatisticalDetector
from src.models.isolation_forest import IsolationForestDetector
from src.models.ensemble import EnsembleDetector
from src.utils.helpers import load_config

plt.style.use('seaborn-v0_8-darkgrid')
config = load_config('../config/config.yaml')

# Load and engineer features
fetcher = DataFetcher(config)
df = fetcher.fetch(ticker='SPY')
fe = FeatureEngineer(config)
featured = fe.engineer(df)
feature_cols = [c for c in fe.get_feature_columns() if c in featured.columns]

print(f"Data: {featured.shape[0]} samples, {len(feature_cols)} features")

## Helper: Plot detections on price chart

Reusable function to visualize any model's detections.

In [ ]:
def plot_detections(dates, prices, scores, anomalies, title, score_label='Anomaly Score'):
    """Plot price with anomaly markers and score subplot."""
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        row_heights=[0.6, 0.4], subplot_titles=['Price + Detections', score_label])
    
    fig.add_trace(go.Scatter(x=dates, y=prices, name='Close',
                  line=dict(color='#2196F3', width=1)), row=1, col=1)
    
    anom_mask = np.array(anomalies, dtype=bool)
    if anom_mask.any():
        fig.add_trace(go.Scatter(x=dates[anom_mask], y=np.array(prices)[anom_mask],
                      mode='markers', name='Anomaly',
                      marker=dict(color='red', size=6, symbol='circle')), row=1, col=1)
    
    fig.add_trace(go.Scatter(x=dates, y=scores, name='Score',
                  line=dict(color='#FF9800', width=1),
                  fill='tozeroy', fillcolor='rgba(255,152,0,0.1)'), row=2, col=1)
    fig.add_hline(y=1.0, line_dash='dash', line_color='red', row=2, col=1)
    
    n_anomalies = int(anom_mask.sum())
    fig.update_layout(height=500, template='plotly_dark', showlegend=True,
                      title=f'{title} — {n_anomalies} anomalies ({n_anomalies/len(dates):.1%})')
    fig.show()

## 1. Naive Baseline

**How it works:** Flag any day where |return| > 2% or volume z-score > 2.5. No learning, no parameters.

**Purpose:** If ML models can't beat this, they aren't adding value over a simple rule.

In [ ]:
naive = NaiveBaselineDetector(config)
naive_result = naive.detect(featured)

plot_detections(featured.index, featured['Close'].values,
               naive_result['anomaly_score'].values,
               naive_result['anomaly'].values,
               'Naive Baseline')

print(f"Anomalies flagged: {naive_result['anomaly'].sum()} / {len(naive_result)} ({naive_result['anomaly'].mean():.1%})")
print(f"\nBreakdown:")
print(f"  Return-triggered:  {(naive_result['return_score'] > 1).sum()}")
print(f"  Volume-triggered:  {(naive_result['volume_score'] > 1).sum()}")

## 2. Statistical Detector (Z-score + EWMA)

**How it works:**
- **Z-score:** measures how many standard deviations a return is from the mean. Flags if |z| > 3.
- **EWMA:** tracks an exponentially weighted moving average and standard deviation. Flags if the value falls outside control limits. Adapts to changing volatility (uses shifted baseline to prevent the current spike from inflating its own bounds).

**Reference:** Shewhart control chart theory; Roberts (1959) "Control Chart Tests Based on Geometric Moving Averages"

In [ ]:
stat = StatisticalDetector(config)
stat_result = stat.detect(featured['returns'])

plot_detections(featured.index, featured['Close'].values,
               stat_result['anomaly_score'].values,
               stat_result['anomaly'].values,
               'Statistical (Z-score + EWMA)')

# Show Z-score vs EWMA contributions
print(f"Z-score flagged:  {stat_result['zscore_score'].gt(1).sum()}")
print(f"EWMA flagged:     {stat_result['ewma_score'].gt(1).sum()}")
print(f"Combined:         {stat_result['anomaly'].sum()}")

# EWMA control chart visualization (zoomed)
ewma_only = stat.ewma_detect(featured['returns'])
zoom = ewma_only['2023-06-01':'2024-01-01']
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(zoom.index, zoom['value'], color='#2196F3', linewidth=0.8, label='Returns')
ax.fill_between(zoom.index, zoom['lower'], zoom['upper'], alpha=0.2, color='orange', label='EWMA Control Limits')
ax.plot(zoom.index, zoom['ewma'], color='orange', linewidth=1, linestyle='--', label='EWMA')
anomalies_zoom = zoom[zoom['anomaly']]
ax.scatter(anomalies_zoom.index, anomalies_zoom['value'], color='red', s=30, zorder=5, label='Anomaly')
ax.set_title('EWMA Control Chart (6-month zoom)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Isolation Forest

**How it works:** Builds an ensemble of random trees that isolate observations by randomly selecting features and split values. Anomalies are isolated in fewer splits (shorter path length) because they are few and different from the majority.

**Key insight (Liu et al., 2008):** Instead of profiling normal data, Isolation Forest directly measures how easy each point is to isolate. This makes it effective in high-dimensional feature spaces.

**Advantages:** No assumption about data distribution, handles mixed feature types, scales well.
**Limitations:** Contamination parameter affects threshold (we use `auto` to avoid leaking info).

In [ ]:
iso = IsolationForestDetector(config)
iso_result = iso.detect(featured, feature_cols=feature_cols)

plot_detections(featured.index, featured['Close'].values,
               iso_result['anomaly_score'].values,
               iso_result['anomaly'].values,
               'Isolation Forest (200 trees)')

# Which features drive isolation?
# Look at feature values of detected anomalies vs normal points
anomaly_mask = iso_result['anomaly']
print("Feature means: Anomalous vs Normal points")
print(f"{'Feature':>20s}  {'Normal':>10s}  {'Anomaly':>10s}  {'Ratio':>8s}")
print("-" * 55)
for col in feature_cols:
    normal_mean = featured.loc[~anomaly_mask, col].mean()
    anom_mean = featured.loc[anomaly_mask, col].mean()
    if abs(normal_mean) > 1e-6:
        ratio = abs(anom_mean / normal_mean)
    else:
        ratio = float('inf')
    print(f"{col:>20s}  {normal_mean:>10.4f}  {anom_mean:>10.4f}  {ratio:>8.2f}x")

## 4. Ensemble

**How it works:** Combines scores from all models using configurable weights:
- Statistical: 25%
- Isolation Forest: 35%  
- Autoencoder: 40% (when available)

Also tracks **consensus** — how many individual models agree a point is anomalous. Points flagged by multiple models are more likely to be true anomalies.

In [ ]:
# Combine statistical + isolation forest scores
n = min(len(stat_result), len(iso_result))
ens = EnsembleDetector(config)
scores = {
    'statistical': stat_result['anomaly_score'].values[-n:],
    'isolation_forest': iso_result['anomaly_score'].values[-n:],
}
ens_result = ens.detect(scores, index=featured.index[-n:])

plot_detections(featured.index[-n:], featured['Close'].values[-n:],
               ens_result['ensemble_score'].values,
               ens_result['anomaly'].values,
               'Ensemble (Statistical + Isolation Forest)')

# Consensus analysis
print("Consensus breakdown:")
print(f"  0 models agree (normal):  {(ens_result['n_votes'] == 0).sum()}")
print(f"  1 model flags:            {(ens_result['n_votes'] == 1).sum()}")
print(f"  2 models agree (strong):  {(ens_result['n_votes'] == 2).sum()}")

# Venn-style overlap
stat_flags = set(stat_result.index[stat_result['anomaly']])
iso_flags = set(iso_result.index[iso_result['anomaly']])
both = stat_flags & iso_flags
stat_only = stat_flags - iso_flags
iso_only = iso_flags - stat_flags
print(f"\nOverlap analysis:")
print(f"  Statistical only: {len(stat_only)}")
print(f"  Isolation Forest only: {len(iso_only)}")
print(f"  Both agree: {len(both)}")
print(f"  Agreement rate: {len(both) / max(len(stat_flags | iso_flags), 1):.1%}")

## 5. Side-by-Side Comparison

Overlay all models' anomaly scores on the same timeline to see where they agree and disagree.

In [ ]:
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.35, 0.22, 0.22, 0.22],
                    subplot_titles=['SPY Close Price', 'Naive Baseline Score',
                                    'Statistical Score', 'Isolation Forest Score'])

fig.add_trace(go.Scatter(x=featured.index, y=featured['Close'],
              line=dict(color='#2196F3', width=1), name='Price'), row=1, col=1)

models = [
    (naive_result['anomaly_score'], '#E91E63', 'Naive'),
    (stat_result['anomaly_score'], '#4CAF50', 'Statistical'),
    (pd.Series(iso_result['anomaly_score'].values, index=featured.index), '#FF9800', 'Isolation Forest'),
]

for i, (scores_series, color, name) in enumerate(models):
    fig.add_trace(go.Scatter(x=scores_series.index, y=scores_series.values,
                  line=dict(color=color, width=0.8), name=name,
                  fill='tozeroy', fillcolor=f'rgba{tuple(int(color.lstrip("#")[j:j+2], 16) for j in (0, 2, 4)) + (0.1,)}'),
                  row=i+2, col=1)
    fig.add_hline(y=1.0, line_dash='dash', line_color='red', row=i+2, col=1)

fig.update_layout(height=800, template='plotly_dark', title='All Models — Score Comparison')
fig.show()

## Key Takeaways

1. **Naive baseline catches obvious events** — it flags big return days directly. This is the minimum bar for any anomaly detector.

2. **Statistical detector is return-focused** — it primarily fires on large daily moves. The EWMA component adapts to volatility regimes, so it flags less during already-volatile periods (good) but can miss the start of a new regime (bad).

3. **Isolation Forest catches multi-dimensional outliers** — it flags points that are unusual across *multiple features simultaneously*, not just extreme returns. This is where it can add value over the baseline.

4. **Models disagree often** — low overlap between statistical and IF detections shows they capture different anomaly types. This is why ensemble makes sense.

5. **Ensemble reduces noise** — requiring consensus from multiple models cuts false positives, but may also increase false negatives (missed anomalies).

**Next:** Honest evaluation on held-out test data with proper metrics.